In [ ]:
pip install torch torchvision timm opencv-python scikit-learn tqdm facenet-pytorch

INFO: pip is looking at multiple versions of facenet-pytorch to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 43.4 MB/s eta 0:00:00


In [ ]:
import os
import cv2
import torch
import timm
import numpy as np

from tqdm import tqdm
from torch import nn
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", DEVICE)

# CHANGE THIS PATH
DATASET_PATH = "./dataset"

FRAME_COUNT = 10
IMG_SIZE = 224
BATCH_SIZE = 4
EPOCHS = 3

Using device: cuda


In [ ]:
import os
import zipfile

print("Checking dataset...")

# Step 1: Mount Drive
from google.colab import drive
drive.mount('/content/drive')

# Step 2: Paths
base_path = "/content/drive/MyDrive/Faceforensics_dataset"
zip_path = os.path.join(base_path, "archive.zip")
extract_path = os.path.join(base_path, "extracted")

# Step 3: Check base folder
if not os.path.exists(base_path):
    raise Exception(f"Base folder not found: {base_path}")

print("Base folder contents:", os.listdir(base_path))

# Step 4: Check zip file
if not os.path.exists(zip_path):
    raise Exception(f"Zip file not found at: {zip_path}")

print("Zip file found!")

# Step 5: Extract (only once)
if not os.path.exists(extract_path):
    print("Extracting dataset...")
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_path)
else:
    print("Dataset already extracted.")

print("Extracted contents:", os.listdir(extract_path))

# Step 6: Go inside FF++ folder
ffpp_path = os.path.join(extract_path, "FF++")

if not os.path.exists(ffpp_path):
    raise Exception("FF++ folder not found after extraction!")

print("FF++ contents:", os.listdir(ffpp_path))

# Step 7: Validate real & fake folders
assert "real" in os.listdir(ffpp_path), "Missing 'real' folder"
assert "fake" in os.listdir(ffpp_path), "Missing 'fake' folder"

print("✅ Dataset is correctly structured!")

# Optional: define final paths
real_path = os.path.join(ffpp_path, "real")
fake_path = os.path.join(ffpp_path, "fake")

print("Real path:", real_path)
print("Fake path:", fake_path)

Checking dataset...
Mounted at /content/drive
Base folder contents: ['archive.zip', 'extracted']
Zip file found!
Dataset already extracted.
Extracted contents: ['FF++']
FF++ contents: ['fake', 'real']
✅ Dataset is correctly structured!
Real path: /content/drive/MyDrive/Faceforensics_dataset/extracted/FF++/real
Fake path: /content/drive/MyDrive/Faceforensics_dataset/extracted/FF++/fake


In [ ]:
from facenet_pytorch import MTCNN

# Force CPU to prevent crashes
mtcnn = MTCNN(
    image_size=224,
    margin=10,
    device="cpu"
)

In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Resize((224,224)),
    transforms.Normalize([0.5]*3, [0.5]*3)
])

In [ ]:
def extract_frames(video_path):
    frames = []
    cap = cv2.VideoCapture(video_path)

    if not cap.isOpened():
        return frames

    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    interval = max(total // FRAME_COUNT, 1)

    for i in range(FRAME_COUNT * 2):  # try more attempts
        cap.set(cv2.CAP_PROP_POS_FRAMES, i * interval)
        ret, frame = cap.read()

        if not ret:
            break

        frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

        try:
            face = mtcnn(frame)
            if face is not None:
                frames.append(face)

            if len(frames) == FRAME_COUNT:
                break

        except:
            continue

    cap.release()
    return frames

In [ ]:
class DeepfakeDataset(Dataset):

    def __init__(self, root):
        self.samples = []

        for label, folder in enumerate(["real", "fake"]):
            folder_path = os.path.join(root, folder)

            for file in os.listdir(folder_path):
                if file.endswith(".mp4"):
                    self.samples.append(
                        (os.path.join(folder_path, file), label)
                    )

        print("Total videos found:", len(self.samples))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        video_path, label = self.samples[idx]

        frames = extract_frames(video_path)

        # 🔥 FIX 1: If no frames detected → full padding
        if len(frames) == 0:
            frames = [torch.zeros(3, 224, 224)] * FRAME_COUNT

        # 🔥 FIX 2: If fewer frames → pad with last frame
        elif len(frames) < FRAME_COUNT:
            pad_count = FRAME_COUNT - len(frames)
            frames += [frames[-1]] * pad_count

        # 🔥 FIX 3: If more frames → trim
        frames = frames[:FRAME_COUNT]

        # 🔥 FIX 4: Safe stacking
        frames = torch.stack(frames)  # (T, C, H, W)

        return frames, torch.tensor(label, dtype=torch.long)

In [ ]:
from torch.utils.data import DataLoader

# ✅ Correct dataset path
dataset_path = "/content/drive/MyDrive/Faceforensics_dataset/extracted/FF++"

dataset = DeepfakeDataset(dataset_path)

train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size

train_dataset, val_dataset = torch.utils.data.random_split(dataset, [train_size, val_size])

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE)

print("✅ DataLoader ready!")

Total videos found: 400
✅ DataLoader ready!


In [ ]:
class DeepfakeDetector(nn.Module):

    def __init__(self):
        super().__init__()

        self.cnn = timm.create_model(
            "efficientnet_b0",
            pretrained=True,
            num_classes=0
        )

        self.gru = nn.GRU(1280, 128, batch_first=True)

        self.fc = nn.Sequential(
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, 2)
        )

    def forward(self, x):
        b, t, c, h, w = x.shape

        x = x.view(b * t, c, h, w)
        x = self.cnn(x)

        x = x.view(b, t, -1)

        _, h_n = self.gru(x)

        out = self.fc(h_n[-1])

        return out

In [ ]:
model = DeepfakeDetector().to(DEVICE)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
criterion = nn.CrossEntropyLoss()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model.safetensors:   0%|          | 0.00/21.4M [00:00<?, ?B/s]

In [ ]:
for epoch in range(EPOCHS):

    model.train()
    running_loss = 0.0

    for frames, labels in tqdm(train_loader):

        # KEEP full sequence → (B, T, C, H, W)
        frames = frames.to(DEVICE)
        labels = labels.to(DEVICE)

        preds = model(frames)

        loss = criterion(preds, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    # ================= EVALUATION =================
    model.eval()

    preds_all = []
    labels_all = []

    with torch.no_grad():

        for frames, labels in val_loader:

            frames = frames.to(DEVICE)
            labels = labels.to(DEVICE)

            outputs = model(frames)

            preds = torch.argmax(outputs, dim=1)

            preds_all.extend(preds.cpu().numpy())
            labels_all.extend(labels.cpu().numpy())

    # ================= METRICS =================
    acc = accuracy_score(labels_all, preds_all)
    f1 = f1_score(labels_all, preds_all, zero_division=0)
    precision = precision_score(labels_all, preds_all, zero_division=0)
    recall = recall_score(labels_all, preds_all, zero_division=0)

    print(f"\nEpoch: {epoch+1}/{EPOCHS}")
    print(f"Loss: {running_loss:.4f}")
    print(f"Accuracy: {acc:.4f}")
    print(f"F1 Score: {f1:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall: {recall:.4f}")

100%|██████████| 80/80 [1:17:48<00:00, 58.35s/it]



Epoch: 1/3
Loss: 49.9898
Accuracy: 0.8500
F1 Score: 0.8537
Precision: 0.7609
Recall: 0.9722


100%|██████████| 80/80 [1:13:03<00:00, 54.79s/it]



Epoch: 2/3
Loss: 31.1218
Accuracy: 0.8625
F1 Score: 0.8675
Precision: 0.7660
Recall: 1.0000


100%|██████████| 80/80 [1:12:58<00:00, 54.73s/it]



Epoch: 3/3
Loss: 17.8973
Accuracy: 0.8625
F1 Score: 0.8493
Precision: 0.8378
Recall: 0.8611


In [ ]:
torch.save(model.state_dict(), "deepfake_model.pth")

In [ ]:
def predict_video():

    model.eval()

    video_path = "/content/drive/MyDrive/Test/test.mp4"

    frames = extract_frames(video_path)

    if len(frames) == 0:
        return "No face detected"

    # Ensure fixed number of frames
    if len(frames) < FRAME_COUNT:
        frames += [frames[-1]] * (FRAME_COUNT - len(frames))

    frames = frames[:FRAME_COUNT]

    processed = []

    for f in frames:
        f = transform(f)
        processed.append(f)

    frames = torch.stack(processed)          # (T, C, H, W)
    frames = frames.unsqueeze(0).to(DEVICE)  # (1, T, C, H, W)

    with torch.no_grad():
        output = model(frames)

    pred = torch.argmax(output, dim=1).item()

    if pred == 0:
        return "REAL VIDEO"
    else:
        return "FAKE / MORPHED VIDEO"

In [ ]:
result = predict_video()
print(result)

NameError: name 'model' is not defined